# High Stats Analysis - From ROOT (Phase B)
Reads the pre-computed cluster points and metadata written by `process_events_to_root.py`
(Phase A) and reproduces every plot and the `summary.txt` that
`Evaluation_BeforeChargeLightMatching_BeforeBeamWindowCut.ipynb` produces, without recomputing selections, KDTree
completeness/purity matching, or category classification.

Output goes to two places, mirroring the exact same directory structure as the original
notebook: PNGs under a `multi_file_plots_from_root_*` tree, and native ROOT TH1D/TH2D
histograms in one output `.root` file (PyROOT's `canvas.Write()` was the original target but
is unusable on this machine today - see the project plan for why, and how to switch later).

Phase A stores the exact per-(true,reco)-pair completeness and purity (completeness on the true
side via `matched_reco_ids`/`matched_reco_completenesses`, purity on the reco side via
`matched_true_ids`/`matched_true_purities`), so the heatmaps and
`DrawTrueClusterWithMatchedReco` use exact values, not an aggregate approximation - see
`root_reconstruction.py`'s `matched_pairs_exact()`.

In [ ]:
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import uproot

from root_reconstruction import RootEventStore
from root_histogram_writer import save_th1, save_th2, save_th1_counts
from phase_b_process_file import process_one_file, completeness_results_per_cluster_like

from completeness_purity_draw import (
    DrawCompletenessVsTrueEnergyPerJob, DrawClusterCompletenessVsTrueEnergyPerJob,
    DrawCompletenessVsTrueEnergy_MatchedPairs_PerJob, DrawPurityVsRecoChargePerJob,
    DrawCompletenessVsPurity_MatchedPairs,
)
from DrawRecoTrueClusters import DrawLabelPerJob, DrawTrueRecoMatchMultiplicity


In [ ]:
# ============================================================================
# CONFIG (mirrors Evaluation_BeforeChargeLightMatching_BeforeBeamWindowCut.ipynb's config cell)
# ============================================================================
ROOT_INPUT_PATH = "/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/processed_events/20260724_151558.root"  # output of process_events_to_root.py
VIEW = "2view"
GHOSTING = True
PLOTBASEDIR = Path("multi_file_plots_from_root_with_deghosting" if GHOSTING
                    else "multi_file_plots_from_root_without_deghosting")

# ========================================================================
# SELECTIVE FILTERING (Optional, for quick tests/debugging)
# ========================================================================
# Set to None to process everything the ROOT file has, or restrict to specific ones.
# Example: TARGET_FILE = "file2", TARGET_EVENT = 8  (only file2, event 8)
TARGET_APA   = None   # Set to "APA0" or "APA1" to process one APA only
TARGET_FILE  = None   # Set to "file1", "file4", etc. to process one file only
TARGET_EVENT = None   # Set to an event number (0, 1, 8, etc.) to process one event only

In [ ]:
# ============================================================================
# _mean_str is the only helper still needed directly in this notebook (Job-Level summary
# text) - completeness_results_per_cluster_like, the heatmap/pair-reconstruction helpers, and
# load_deadarea_polygons now live in phase_b_process_file.py, alongside the per-file
# plotting pipeline that uses them (see that module's docstring for why: they run inside
# per-file worker processes now, not in this notebook's own process).
# ============================================================================

def _mean_str(values):
    return f"{np.mean(values):.4f}" if len(values) else "N/A"


In [ ]:
# ============================================================================
# SETUP
# ============================================================================
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

store = RootEventStore(ROOT_INPUT_PATH)
apa_list = sorted(store.true_cluster_metadata_df["apa"].unique())
if TARGET_APA is not None:
    apa_list = [a for a in apa_list if a == TARGET_APA]
print(f"APAs found in ROOT file: {apa_list}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir_view = PLOTBASEDIR / VIEW
root_out_path = output_dir_view / f"analysis_{timestamp}.root"
output_dir_view.mkdir(parents=True, exist_ok=True)
root_out = uproot.recreate(root_out_path)
print(f"PNG output base: {output_dir_view}")
print(f"ROOT histogram output: {root_out_path}")

total_events_processed = 0
total_files_processed = 0

In [ ]:
for apa in apa_list:
    print(f"\n{'#'*70}\nPROCESSING {apa}\n{'#'*70}\n")
    output_dir = output_dir_view / f"apa_{apa}_{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=True)
    root_apa_dir = f"apa_{apa}"

    job_true_rows, job_reco_rows, job_pair_metadata_list = [], [], []
    all_histogram_records = []

    all_files = store.files_for_apa(apa)
    files_to_run = [f for f in all_files if TARGET_FILE is None or f == TARGET_FILE]
    for f in all_files:
        if f not in files_to_run:
            print(f"Skipping {f} (target: {TARGET_FILE})")

    # ========================================================================
    # PER-FILE DISPATCH (parallel - see phase_b_process_file.py's docstring: profiling
    # showed Phase B's time is ~70% matplotlib rendering, so parallelizing across files
    # gives a near-linear speedup bounded by CPU core count. File-Level plots run inside
    # each file's own worker (they only need that file's events); Job-Level plots below
    # stay serial since they need every file's results first - a real aggregation barrier,
    # but a small, fixed-size cost that doesn't scale with event count.)
    # ========================================================================
    print(f"\nDispatching {len(files_to_run)} file(s) across up to {os.cpu_count()} worker processes...")
    with ProcessPoolExecutor() as executor:
        futures = {}
        for file_name in files_to_run:
            file_output_dir = output_dir / file_name
            root_file_dir = f"{root_apa_dir}/{file_name}"
            future = executor.submit(process_one_file, ROOT_INPUT_PATH, file_name, apa, VIEW,
                                      file_output_dir, root_file_dir, TARGET_EVENT)
            futures[future] = file_name

        for future in as_completed(futures):
            file_name = futures[future]
            result = future.result()
            print(f"  FILE {file_name}: {result['total_events_processed']} events processed")
            job_true_rows.extend(result["file_true_rows"])
            job_reco_rows.extend(result["file_reco_rows"])
            job_pair_metadata_list.extend(result["file_pair_metadata_list"])
            all_histogram_records.extend(result["histogram_records"])
            total_events_processed += result["total_events_processed"]
            total_files_processed += 1

    # ---- replay every worker's deferred ROOT-histogram writes (root_out can only be
    # written from this one process - see phase_b_process_file.py's docstring) ----
    for rec in all_histogram_records:
        if rec["kind"] == "th2":
            save_th2(root_out, rec["dir"], rec["name"], rec["x"], rec["y"], rec["xbins"], rec["ybins"])
        elif rec["kind"] == "th1":
            save_th1(root_out, rec["dir"], rec["name"], rec["values"], rec["bins"])
        elif rec["kind"] == "th1_counts":
            save_th1_counts(root_out, rec["dir"], rec["name"], rec["counts"])

    # ========================================================================
    # JOB-LEVEL AGGREGATION
    # ========================================================================
    print(f"\nJOB-LEVEL AGGREGATION ({apa}): {total_files_processed} files, {total_events_processed} events")
    if job_true_rows:
        job_agg_output_dir = output_dir / "job_summary"
        job_agg_output_dir.mkdir(parents=True, exist_ok=True)
        root_job_summary_dir = f"{root_apa_dir}/job_summary"

        job_completeness_2d1d_imaging_dir = job_agg_output_dir / "completeness" / "completeness_2d_1d_imaging_level"
        job_completeness_2d1d_clustering_dir = job_agg_output_dir / "completeness" / "completeness_2d_1d_clusteringlevel"
        job_completeness_2d1d_clustering_pairs_only_dir = job_agg_output_dir / "completeness" / "completeness_2d_1d_clusteringlevel_true_reco_pairs_only"
        job_completeness_2d1d_imaging_dir.mkdir(parents=True, exist_ok=True)
        job_completeness_2d1d_clustering_dir.mkdir(parents=True, exist_ok=True)
        job_completeness_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

        job_purity_summary_dir = job_agg_output_dir / "purity"
        job_purity_summary_dir.mkdir(parents=True, exist_ok=True)

        job_eff_vs_purity_incl_dir = job_agg_output_dir / "true_reco_matched_pair_completeness_purity_including_unmatched_true_clusters"
        job_eff_vs_purity_excl_dir = job_agg_output_dir / "true_reco_matched_pair_completeness_purity_excluding_unmatched_true_clusters"
        job_eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
        job_eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

        job_eff_like = completeness_results_per_cluster_like(job_true_rows)
        imaging_energies, imaging_completenesses = DrawCompletenessVsTrueEnergyPerJob(
            job_eff_like, job_completeness_2d1d_imaging_dir, apa, job_metadata_list=job_true_rows)
        DrawClusterCompletenessVsTrueEnergyPerJob(job_pair_metadata_list, job_completeness_2d1d_clustering_dir, apa, all_true_metadata_list=job_true_rows)
        clustering_pairs_energies, clustering_pairs_completenesses = DrawCompletenessVsTrueEnergy_MatchedPairs_PerJob(
            job_pair_metadata_list, job_completeness_2d1d_clustering_pairs_only_dir, apa)
        DrawPurityVsRecoChargePerJob(job_pair_metadata_list, job_purity_summary_dir, apa)
        DrawCompletenessVsPurity_MatchedPairs(job_pair_metadata_list, job_eff_vs_purity_incl_dir, 'Job Level', apa, all_true_metadata_list=job_true_rows)
        DrawCompletenessVsPurity_MatchedPairs(job_pair_metadata_list, job_eff_vs_purity_excl_dir, 'Job Level', apa, all_true_metadata_list=None)
        DrawLabelPerJob(job_true_rows, job_agg_output_dir, apa)
        DrawTrueRecoMatchMultiplicity(job_true_rows, job_agg_output_dir, apa, 'Job Level', 'job')

        print(f"\nJob-level summary statistics:")
        print(f"  Mean Completeness: {_mean_str(imaging_completenesses)}")
        print(f"  Mean Purity: {_mean_str([p['purity'] for p in job_pair_metadata_list])}")

        # ---- mirrored ROOT histograms (job level, "All Clusters") ----
        save_th1(root_out, f"{root_job_summary_dir}/completeness/completeness_2d_1d_imaging_level",
                 "completeness_vs_true_energy_1d_job", imaging_completenesses, 20)
        save_th2(root_out, f"{root_job_summary_dir}/completeness/completeness_2d_1d_imaging_level",
                 "completeness_vs_true_energy_2d_job", imaging_energies, imaging_completenesses, 50, 50)
        save_th1(root_out, f"{root_job_summary_dir}/completeness/completeness_2d_1d_clusteringlevel_true_reco_pairs_only",
                 "completeness_vs_true_energy_1d_clusteringlevel_pairs_only_job", clustering_pairs_completenesses, 20)
        if job_pair_metadata_list:
            job_charges = np.array([r["total_reco_charge"] for r in job_pair_metadata_list])
            job_purities = np.array([r["purity"] for r in job_pair_metadata_list])
            save_th1(root_out, f"{root_job_summary_dir}/purity", "purity_vs_reco_charge_1d_job", job_purities, 20)
            save_th2(root_out, f"{root_job_summary_dir}/purity", "purity_vs_reco_charge_2d_job", job_charges, job_purities, 50, 50)
            job_effs_pairs = np.array([r["completeness"] for r in job_pair_metadata_list])
            save_th2(root_out, f"{root_job_summary_dir}/true_reco_matched_pair_completeness_purity_including_unmatched_true_clusters",
                     "completeness_vs_purity_job", job_effs_pairs, job_purities, 50, 50)

        neutrino_n = sum(1 for r in job_true_rows if r["cluster_type"] == "neutrino")
        cosmic_n = sum(1 for r in job_true_rows if r["cluster_type"] == "cosmic")
        save_th1_counts(root_out, root_job_summary_dir, "true_clusters_by_type_job", [neutrino_n, cosmic_n])

        # ================================================================
        # summary.txt (mirrors the notebook's fixed mean-completeness logic)
        # ================================================================
        energy_threshold_mev = 500.0
        eff_below_500 = [eff for en, eff in zip(imaging_energies, imaging_completenesses) if en < energy_threshold_mev]
        eff_above_500 = [eff for en, eff in zip(imaging_energies, imaging_completenesses) if en >= energy_threshold_mev]
        pair_eff_below_500 = [eff for en, eff in zip(clustering_pairs_energies, clustering_pairs_completenesses) if en < energy_threshold_mev]
        pair_eff_above_500 = [eff for en, eff in zip(clustering_pairs_energies, clustering_pairs_completenesses) if en >= energy_threshold_mev]
        pair_pur_below_500 = [m['purity'] for m in job_pair_metadata_list if m.get('total_true_energy', 0) < energy_threshold_mev]
        pair_pur_above_500 = [m['purity'] for m in job_pair_metadata_list if m.get('total_true_energy', 0) >= energy_threshold_mev]

        summary_lines = [
            "=" * 80, "JOB SUMMARY (from ROOT file - Phase B analysis)", "=" * 80,
            f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"ROOT input: {ROOT_INPUT_PATH}", f"APA: {apa}", "",
            "=" * 80, "JOB-LEVEL AGGREGATION", "=" * 80,
            f"Total files processed: {total_files_processed}",
            f"Total events processed: {total_events_processed}", "",
            "Job-level summary statistics: Imaging Level",
            f"  Mean Completeness (overall):             {_mean_str(imaging_completenesses)}",
            f"  Mean Completeness (< {energy_threshold_mev:.0f} MeV):          {_mean_str(eff_below_500)}  ({len(eff_below_500)} entries)",
            f"  Mean Completeness (>= {energy_threshold_mev:.0f} MeV):         {_mean_str(eff_above_500)}  ({len(eff_above_500)} entries)",
            "=" * 80, "",
            "Job-level summary statistics: Clustering Level",
            f"  Mean Completeness (overall):             {_mean_str(clustering_pairs_completenesses)}",
            f"  Mean Completeness (< {energy_threshold_mev:.0f} MeV):          {_mean_str(pair_eff_below_500)}  ({len(pair_eff_below_500)} entries)",
            f"  Mean Completeness (>= {energy_threshold_mev:.0f} MeV):         {_mean_str(pair_eff_above_500)}  ({len(pair_eff_above_500)} entries)",
            f"  Mean Purity (overall):                  {_mean_str([m['purity'] for m in job_pair_metadata_list])}",
            f"  Mean Purity (< {energy_threshold_mev:.0f} MeV):               {_mean_str(pair_pur_below_500)}  ({len(pair_pur_below_500)} entries)",
            f"  Mean Purity (>= {energy_threshold_mev:.0f} MeV):              {_mean_str(pair_pur_above_500)}  ({len(pair_pur_above_500)} entries)",
            "=" * 80,
            "",
            "=" * 80,
            "JOB RUNTIME",
            "=" * 80,
            f"Job started at: {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}",
            f"Job finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"Total job runtime: {timedelta(seconds=int(time.time() - job_start_time))} ({time.time() - job_start_time:.1f} seconds)",
            "=" * 80,
        ]
        with open(job_agg_output_dir / "summary.txt", "w") as sf:
            sf.write("\n".join(summary_lines) + "\n")
        print(f"Job summary saved to: {job_agg_output_dir / 'summary.txt'}")


In [ ]:
job_elapsed = time.time() - job_start_time
print(f"\nTotal runtime: {timedelta(seconds=int(job_elapsed))} ({job_elapsed:.1f}s)")

root_out.close()
print(f"\nROOT histograms written to: {root_out_path}")